In [ ]:
# @title 1.Lip Sync with Wav2Lip image to lipsync
%cd /content/
# %%
# Cell 1: Mount Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

# %%
# Cell 2: Install system dependencies and Python packages
!apt-get update && apt-get install -y ffmpeg
!pip install -q youtube-dl
!pip install -q opencv-python==4.10.0.84
!pip install -q librosa==0.8.1
!pip uninstall -y numpy
!pip install -q numpy==1.23.5

# %%
# Cell 3: Clone the Wav2Lip repository and install requirements
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -q -r requirements.txt
%cd /content/Wav2Lip/
# %%
# Cell 4: Download pre-trained Wav2Lip model checkpoint
!mkdir -p /checkpoints
!gdown --id 1_OvqStxNxLc7bXzlaVG5sz695p-FVfYY -O checkpoints/wav2lip.pth

%cd /content/Wav2Lip/
# %%
# Cell 5: Upload your input files (face image and audio)
from google.colab import files
uploaded = files.upload()
# Ensure your image is named e.g. 'face.png' and audio 'speech.wav'

# %%
# Cell 6: Run lip-sync inference
!python inference.py \
    --checkpoint_path checkpoints/wav2lip.pth \
    --face face.png \
    --audio speech.wav \
    --outfile result.mp4 \
    --static True


# %%
# Cell 7: Display the resulting video inline
from IPython.display import HTML
from base64 import b64encode
mp4 = open('result.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""<video width=512 controls>
<source src=\"{data_url}\" type=\"video/mp4\">
</video>""")

In [ ]:
# @title 2.video to lipsync with Wav2Lip
# =============================================================================
# Cell 1: Install dependencies
# =============================================================================
%cd /content/
!apt-get update -y && apt-get install -y ffmpeg
!pip install -q youtube-dl numpy==1.23.5 librosa==0.8.1 opencv-python==4.10.0.84 moviepy

# =============================================================================
# Cell 2: Clone Wav2Lip & download model
# =============================================================================
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -q -r requirements.txt
!mkdir -p /checkpoints
!gdown --id 1_OvqStxNxLc7bXzlaVG5sz695p-FVfYY -O checkpoints/wav2lip.pth

# =============================================================================
# Cell 3: Upload your input files
# =============================================================================
from google.colab import files
uploaded = files.upload()
# Expect: one video (mp4/mov/avi/mkv) + one .wav

# Dynamically pick filenames
video_path = next(f for f in uploaded if f.lower().endswith(('.mp4','.mov','.avi','.mkv')))
audio_path = next(f for f in uploaded if f.lower().endswith(('.wav', '.mp3')))

# --- Insert this at the end of Cell 3: sanitize filenames ---
import os, shutil

safe_vid = "input_video.mp4"
safe_wav = "input_audio.wav"

shutil.move(video_path, safe_vid)   # rename the uploaded video
shutil.move(audio_path, safe_wav)   # rename the uploaded wav

# now override the paths you'll use later:
video_path = safe_vid
audio_path = safe_wav

print(f"Renamed video → {video_path}")
print(f"Renamed audio → {audio_path}")


# =============================================================================
# Cell 4: Transcode to a clean MP4
# =============================================================================
import subprocess
clean_vid = "clean_input.mp4"
subprocess.run([
    "ffmpeg","-y",
    "-i", video_path,
    "-c:v","libx264","-preset","veryfast",
    "-c:a","aac",
    "-movflags","+faststart",
    clean_vid
], check=True)
# We’ll loop & reverse 'clean_input.mp4'
in_vid = clean_vid
in_wav = audio_path

# =============================================================================
# Cell 5: Loop, reverse, trim & mux—all in FFmpeg
# =============================================================================
import subprocess, math

# Helper to probe duration
def ffprobe_duration(path):
    out = subprocess.check_output([
        "ffprobe","-v","error",
        "-show_entries","format=duration",
        "-of","default=noprint_wrappers=1:nokey=1",
        path
    ])
    return float(out.strip())

try:
    vid_dur = ffprobe_duration(in_vid)
except subprocess.CalledProcessError:
    raise RuntimeError(f"Cannot read duration of {in_vid}")

aud_dur = ffprobe_duration(in_wav)

# 1) make a reversed copy
rev_vid = "rev_input.mp4"
subprocess.run([
    "ffmpeg","-y",
    "-i", in_vid,
    "-vf","reverse",
    "-af","areverse",
    rev_vid
], check=True)

# 2) compute how many cycles (forward+reverse) we need
cycle = vid_dur * 2
n_cycles = math.ceil(aud_dur / cycle)

# 3) write a concat list file
concat_list = "concat_list.txt"
with open(concat_list, "w") as f:
    for _ in range(n_cycles):
        f.write(f"file '{in_vid}'\n")
        f.write(f"file '{rev_vid}'\n")

# 4) concat, trim to audio length, mux audio
looped = "looped_input.mp4"
subprocess.run([
    "ffmpeg","-y",
    "-f","concat","-safe","0","-i", concat_list,
    "-i", in_wav,
    "-t", str(aud_dur),
    "-c:v","libx264","-preset","veryfast",
    "-c:a","aac",
    "-map","0:v:0","-map","1:a:0",
    looped
], check=True)

print(f"Built looped video → {looped} ({aud_dur:.2f}s)")

# =============================================================================
# Cell 6: Run Wav2Lip inference
# =============================================================================
!python inference.py \
    --checkpoint_path checkpoints/wav2lip.pth \
    --face looped_input.mp4 \
    --audio {in_wav} \
    --outfile result.mp4

# =============================================================================
# Cell 7: Preview the result
# =============================================================================
from IPython.display import HTML
from base64 import b64encode

mp4 = open('result.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""<video width=512 controls>
  <source src="{data_url}" type="video/mp4">
</video>""")
